# DAT09 · 미션9 변환 규칙을 고치고 별도 자료로 확인하기 (교수자·해설용 워크북 — 예시 풀이 포함)

- **예상 실습 시간**: 3시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_reference`
- **연결 파일**: 첫 셀에서 `DAT09.md`를 연결하세요.
- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션8을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m08.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

> 학습자 배포 시에는 `TODO(n, step)` 상태인 `DAT09_m09_student.ipynb`만 배포하고, 본 워크북은 해설·채점 참고용으로 분리 보관하세요.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'solution'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=8):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 8)
else:
    print('DAT09.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션9 변환 규칙을 고치고 별도 자료로 확인하기

예상 3시간입니다.

#### 이번 미션의 목표

허용된 형식 변화만 변환 규칙에 추가하고, 개발에 쓰지 않은 자료와 사용 확인 시험으로 개선 및 되돌리기를 검증합니다.

### 1단계 기존 규칙이 다섯 형식 변화를 어떻게 처리하는지 재현하기

#### 이 단계에서 하는 일

새 파일을 읽지 못했다고 모두 잘못된 보고인 것은 아닙니다. 허용할 수 있는 표기 변화와 새 품목처럼 승인이 필요한 변화를 구별해야 합니다. 먼저 수정 전 규칙의 결과를 남겨 비교 기준을 만듭니다.

#### 시작 전 확인

미션8까지 완료한 결과를 사용합니다. 노트북의 m09_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/aar_dev.json의 개발 70건과 그 안의 training_expectation을 사용합니다. 이는 개발용 예상 결과이며 별도 평가 30건의 정답과 구분됩니다.

#### 작업 순서

- v1 매핑 함수를 작성하세요. 품목은 S 뒤 세 자리 숫자, 거점은 DEP 뒤 세 자리 숫자만 허용하고 등록된 코드인지 확인합니다. 날짜는 시간대 있는 ISO 형식, 결측 표시는 공란·NULL·N/A를 사용합니다.

- 개발 70건에 v1을 적용하고 case_id·상태·변환 품목·거점·수량·날짜·실패 이유를 저장하세요.

- 품목 자릿수 S1, 거점 별칭 DEPOT-1, 점으로 구분한 날짜, “미집계” 수량, 새 품목 S901의 다섯 유형을 각각 찾아 실패하거나 보류된 이유를 적으세요.

- 새 품목은 표기 오류와 다릅니다. 숫자 형태를 읽을 수 있어도 카탈로그에 없는 품목을 기존 품목으로 추측해 연결하지 않습니다.


#### 저장할 결과

aar_v1_cases.csv, root_cause.md와 c['mapping_fn'], c['aar_dev'], c['aar_v1']를 남깁니다.

#### 직접 확인하는 방법

70건이 모두 결과에 있고 다섯 유형별 대표 사례를 찾아볼 수 있어야 합니다. 정상 형식을 기존 규칙이 그대로 받아들이는지도 확인하세요.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M09/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M09_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 검토한 형식만 v2 규칙에 추가하기

#### 이 단계에서 하는 일

오류를 줄이려고 모든 문자열을 비슷한 코드에 연결하면 엉뚱한 품목의 재고가 됩니다. 허용한 표기와 등록 범위를 명시하고, 승인되지 않은 새 품목은 계속 보류합니다.

#### 시작 전 확인

미션9의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m09_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

v1 함수, 개발 자료와 catalog.json을 사용합니다. 수정 내용은 config/mapping_v2.json으로 따로 저장해 되돌릴 수 있게 합니다.

#### 작업 순서

- S 뒤 1~3자리 숫자를 읽어 세 자리 공통 품목으로 바꾸되 등록된 품목만 연결하세요. DEP 세 자리와 DEPOT-숫자 별칭도 등록 거점 안에서만 허용합니다.

- 시간대 있는 점 구분 날짜 형식 %Y.%m.%d %H:%M%z와 결측 표시 “미집계”를 추가하세요. 날짜의 시간대를 추정하거나 결측을 숫자 0으로 바꾸지 않습니다.

- S901 같은 미등록 품목은 HELD와 PENDING_NEW_KEY로 남기고 item_id를 확정하지 마세요. 문자열 유사도로 기존 품목을 고르는 처리는 넣지 않습니다.

- 같은 개발 70건에 v2를 실행하고 변경한 조건·허용 범위·버전·보류 정책을 patch_diff.md와 설정에 적으세요. v1 결과와 원자료는 보존합니다.


#### 저장할 결과

config/mapping_v2.json, patch_diff.md와 v2의 70건 결과를 남깁니다.

#### 직접 확인하는 방법

S1은 등록된 I001로 연결되고 DEPOT-1은 U001로 연결돼야 합니다. “미집계”는 결측 보류이며 S901은 기존 품목으로 연결되지 않아야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M09/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M09_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 해결한 누락과 잘못 연결한 보고를 따로 계산하기

#### 이 단계에서 하는 일

수용 건수가 늘었다는 이유만으로 개선됐다고 볼 수 없습니다. 잘못된 품목을 정상으로 연결한 건이 늘었을 수도 있습니다. 올바른 연결을 더 찾았는지와 잘못 연결한 것이 없는지를 함께 계산합니다.

#### 시작 전 확인

미션9의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m09_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

같은 개발 70건의 v1·v2 결과와 training_expectation을 사용합니다. 한 버전에 다른 입력을 주면 비교가 성립하지 않습니다.

#### 작업 순서

- 정상으로 연결한 보고 중 예상 품목·거점·수용 상태가 모두 맞는 건을 TP로 세세요. 정밀도는 올바른 수용/전체 수용, 재현율은 올바른 수용/원래 수용해야 할 건수입니다. F1은 두 비율의 조화평균입니다.

- 틀린 품목·거점 또는 수용하면 안 되는 보고를 수용한 수를 wrong_links로 세세요. 자릿수·거점 별칭·날짜 형식 중 해결할 수 있는데 수용하지 못한 수를 resolvable_missing으로 셉니다.

- v1에서 정상 연결한 보고가 v2에서도 같은 품목·거점으로 수용되는지 확인하세요. 신규 품목의 정당한 보류와 해결 가능한 누락을 섞지 않습니다.

- 버전별 수용·보류·격리·정밀도·재현율·F1·오연결·해결 가능 누락을 같은 표에 놓고 바뀐 사례 ID를 확인하세요.


#### 저장할 결과

aar_comparison.csv와 정상 보고 보존·변경 사례의 대조 기록을 저장합니다.

#### 직접 확인하는 방법

v2에서 해결 가능한 누락 0건, 오연결 0건이어야 합니다. 새 품목과 결측이 보류돼 수용률이 100%가 아니어도 올바른 결과일 수 있습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M09/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M09_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 개발 자료로 기준을 정한 뒤 별도 자료로 평가하기

#### 이 단계에서 하는 일

평가 결과를 보면서 규칙을 계속 고치면 처음 보는 자료에 대한 성능을 알 수 없습니다. 기준을 정하는 개발 자료와 마지막에 한 번 확인하는 평가 자료를 분리합니다.

#### 시작 전 확인

미션9의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m09_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

alert_dev.json의 경보 개발 자료, aar_holdout.json의 매핑 평가 30건과 alert_holdout.json의 별도 경보 사례를 사용합니다. holdout은 규칙 수정에 사용하지 않은 평가 자료입니다.

#### 작업 순서

- 경보 개발 자료에서 수량 차이 비율 기준 0·2·5%와 지연 기준 30·60·120분의 아홉 조합을 시험하세요. 차이 비율이 기준보다 크거나 지연이 기준보다 크면 경보를 냅니다.

- 각 조합의 TP·FP·FN·F1을 계산하고 최고 F1을 선택하세요. 동점이면 FP가 적은 조합을 고르는 등 동점 처리도 미리 정합니다.

- 선택한 경보 기준과 v2 매핑을 고정한 뒤 평가 30건과 별도 경보 자료의 예측을 frozen_predictions.json에 먼저 저장하세요.

- 그다음 교관 정답표로 평가하세요. 평가 결과를 보고 기준을 다시 바꾼 경우 이번 자료는 더 이상 독립 평가로 부르지 말고 변경 기록과 새 평가 필요를 남깁니다. 정답표가 없으면 미채점입니다.


#### 저장할 결과

tuning_results.csv, frozen_predictions.json, holdout_results.csv와 alert_holdout_results.json을 저장합니다.

#### 직접 확인하는 방법

개발 조합은 아홉 개, 매핑 평가 결과는 30건이어야 합니다. 예측 저장이 정답 확인보다 먼저인지 코드 순서로 확인하세요.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M09/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M09_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 스무 항목의 사용 확인과 규칙 되돌리기 시험하기

#### 이 단계에서 하는 일

개선한 규칙은 새 사례만 잘 읽는 것으로 끝나지 않습니다. 이전 기능을 깨뜨리지 않았는지 확인하고 문제가 생기면 이전 버전으로 돌아가 같은 결과를 재현할 수 있어야 합니다.

#### 시작 전 확인

미션9의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m09_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/uat_checks.json의 20항목, 앞 미션의 시험 근거와 v1·v2 결과를 사용합니다. UAT는 사용할 사람이 요구한 동작을 확인하는 시험이며 각 항목은 5점입니다.

#### 작업 순서

- 20항목을 빠짐없이 읽고 해당 기능을 실제로 실행한 근거와 연결하세요. 실행해 맞으면 PASS, 실행했지만 다르면 FAIL, 실행·정답 확인을 못 했으면 NOT_RUN으로 기록합니다.

- 각 항목에 5점 또는 0점을 주고 총점을 계산하세요. critical=true인 항목은 핵심 조건이므로 한 개라도 FAIL·NOT_RUN이면 인수 준비 완료로 판정하지 않습니다.

- 원자료를 그대로 둔 채 v1 규칙을 다시 적용하고 최초 v1의 70건 결과와 case_id별 상태·코드·값을 비교하세요. 버전 문자열만 v1으로 바꾸는 것으로 대신하지 않습니다.

- 다시 v2를 적용할 방법, 보관할 설정·입력 해시·결과, 미채점 항목과 보완 담당자를 rollback.md에 적으세요.


#### 저장할 결과

uat_results.csv와 rollback.md에 20항목 판정·점수·구체적 근거 경로·되돌리기 결과를 저장합니다.

#### 직접 확인하는 방법

정답과 시험 근거가 모두 있는 기준 실행은 100점입니다. 학습자 실행은 실제 수행한 결과로 채점하며 미실행을 통과로 채우지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M09/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M09_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 기존 규칙이 다섯 형식 변화를 어떻게 처리하는지 재현하기
def m09_s1(c):
    """기존 규칙이 다섯 형식 변화를 어떻게 처리하는지 재현하기."""
    out = folder(c, 9)
    cat = c['catalog']
    item_map = cat['mapping']['A']['item']
    site_map = cat['mapping']['A']['site']

    def mapping_fn(case, mode='v1'):
        ic = str(case.get('item_code', ''))
        sc = str(case.get('site_code', ''))
        qty = case.get('quantity')
        as_of = str(case.get('as_of', ''))
        reasons = []
        mapped_item = None
        mapped_site = None

        if mode == 'v1':
            if re.fullmatch(r'S\d{3}', ic) and ic in item_map:
                mapped_item = item_map[ic]
            else:
                reasons.append('ITEM_FORMAT_OR_UNKNOWN')
            if re.fullmatch(r'DEP\d{3}', sc) and sc in site_map:
                mapped_site = site_map[sc]
            else:
                reasons.append('SITE_FORMAT_OR_UNKNOWN')
            missing_tokens = {'', 'NULL', 'N/A'}
            date_ok = False
            try:
                dt_obj = datetime.fromisoformat(as_of.replace('Z', '+00:00'))
                date_ok = dt_obj.tzinfo is not None
            except Exception:
                date_ok = False
            if not date_ok:
                reasons.append('DATE_FORMAT_ERROR')
        else:
            # v2: 등록 범위 안에서만 자릿수 패딩 및 DEPOT-숫자 별칭 허용, 신규 키는 HELD/PENDING_NEW_KEY
            m_item = re.fullmatch(r'S(\d{1,3})', ic)
            if m_item:
                cand_ic = f"S{int(m_item.group(1)):03d}"
                if cand_ic in item_map:
                    mapped_item = item_map[cand_ic]
                else:
                    reasons.append('PENDING_NEW_KEY')
            else:
                reasons.append('ITEM_FORMAT_ERROR')

            m_site = re.fullmatch(r'(?:DEP|DEPOT-)(\d{1,3})', sc)
            if m_site:
                cand_sc = f"DEP{int(m_site.group(1)):03d}"
                if cand_sc in site_map:
                    mapped_site = site_map[cand_sc]
                else:
                    reasons.append('PENDING_NEW_KEY')
            else:
                reasons.append('SITE_FORMAT_ERROR')

            missing_tokens = {'', 'NULL', 'N/A', '미집계'}
            date_ok = False
            for fmt_fn in (
                lambda s: datetime.fromisoformat(s.replace('Z', '+00:00')),
                lambda s: datetime.strptime(s, '%Y.%m.%d %H:%M%z'),
            ):
                try:
                    dt_obj = fmt_fn(as_of)
                    if dt_obj.tzinfo is not None:
                        date_ok = True
                        break
                except Exception:
                    continue
            if not date_ok:
                reasons.append('DATE_FORMAT_ERROR')

        is_missing = (qty is None) or (isinstance(qty, str) and qty.strip() in missing_tokens)
        if is_missing:
            reasons.append('MISSING_QTY')

        if 'DATE_FORMAT_ERROR' in reasons or 'ITEM_FORMAT_ERROR' in reasons or 'SITE_FORMAT_ERROR' in reasons:
            status = 'QUARANTINED'
        elif 'PENDING_NEW_KEY' in reasons or 'MISSING_QTY' in reasons or (mode == 'v1' and ('ITEM_FORMAT_OR_UNKNOWN' in reasons or 'SITE_FORMAT_OR_UNKNOWN' in reasons)):
            status = 'HELD'
        else:
            status = 'ACCEPTED'
        return {
            'case_id': case['case_id'], 'mode': mode, 'status': status,
            'item_id': mapped_item, 'site_id': mapped_site,
            'quantity': None if is_missing else qty, 'as_of': as_of,
            'reasons': '|'.join(reasons) if reasons else 'OK',
        }

    aar_dev = read_json(c['root'] / 'data/aar_dev.json')
    v1_rows = [mapping_fn(r, mode='v1') for r in aar_dev]
    csv_out(out / 'aar_v1_cases.csv', v1_rows)
    (out / 'root_cause.md').write_text(
        '# M09 AAR v1 근본 원인 분석\n'
        '- `padding`(S1 등): S 뒤 3자리 고정 정규식으로 실패 -> 등록 범위 내 1~3자리 패딩 허용 검토.\n'
        '- `site_alias`(DEPOT-1 등): DEP3자리만 허용해 실패 -> 등록 거점 내 DEPOT-숫자 별칭 허용.\n'
        '- `date_format`(YYYY.MM.DD HH:MM+0900): ISO8601만 허용해 실패 -> 시간대 포함 점 구분 형식 추가.\n'
        '- `missing_token`("미집계"): 결측 토큰 목록에 없어 처리 누락 -> 0으로 채우지 않고 HELD로 분류.\n'
        '- `new_item`(S901): 카탈로그 미등록 신규 품목 -> 추측 연결 금지, HELD(PENDING_NEW_KEY) 유지.\n',
        encoding='utf-8',
    )
    c['aar_dev'] = aar_dev
    c['aar_v1'] = v1_rows
    c['mapping_fn'] = mapping_fn
    return v1_rows

# STEP 02 검토한 형식만 v2 규칙에 추가하기
def m09_s2(c):
    """검토한 형식만 v2 규칙에 추가하기."""
    out = folder(c, 9)
    v2_cfg = {
        'version': '2.0',
        'catalog': 'data/catalog.json',
        'item_pattern': 'S followed by 1..3 digits zero-padded to S###; known catalog only',
        'site_pattern': 'DEP### or DEPOT-# mapped to DEP###; known catalog only',
        'date_formats': ['ISO8601 with timezone', '%Y.%m.%d %H:%M%z'],
        'missing_tokens': ['', 'NULL', 'N/A', '미집계'],
        'unregistered_key_policy': 'HELD with PENDING_NEW_KEY',
    }
    write_json(c['root'] / 'config/mapping_v2.json', v2_cfg)
    (out / 'patch_diff.md').write_text(
        '# mapping v1 -> v2 변경 내역\n'
        '- 허용 변화: S1~S020 자릿수 패딩, DEPOT-1~6 거점 별칭, `%Y.%m.%d %H:%M%z` 날짜, `"미집계"` 결측 토큰.\n'
        '- 유지 정책: 미등록 키(`S901` 등)는 유사도 추측 연결 없이 `HELD / PENDING_NEW_KEY`로 보류.\n',
        encoding='utf-8',
    )
    v2_rows = [c['mapping_fn'](r, mode='v2') for r in c['aar_dev']]
    c['aar_v2'] = v2_rows
    return v2_rows

# STEP 03 해결한 누락과 잘못 연결한 보고를 따로 계산하기
def m09_s3(c):
    """해결한 누락과 잘못 연결한 보고를 따로 계산하기."""
    out = folder(c, 9)
    dev_exp = {r['case_id']: r['training_expectation'] for r in c['aar_dev']}

    def eval_ver(ver, rows):
        acc = sum(1 for r in rows if r['status'] == 'ACCEPTED')
        held = sum(1 for r in rows if r['status'] == 'HELD')
        quar = sum(1 for r in rows if r['status'] == 'QUARANTINED')
        should_acc = sum(1 for e in dev_exp.values() if e['expected_status'] == 'ACCEPTED')
        tp = 0
        wrong_links = 0
        resolvable_missing = 0
        for r in rows:
            exp = dev_exp[r['case_id']]
            if r['status'] == 'ACCEPTED':
                if exp['expected_status'] == 'ACCEPTED' and r['item_id'] == exp['expected_item_id'] and r['site_id'] == exp['expected_site_id']:
                    tp += 1
                else:
                    wrong_links += 1
            elif exp['expected_status'] == 'ACCEPTED':
                resolvable_missing += 1
        prec = tp / acc if acc else 0.0
        rec = tp / should_acc if should_acc else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        return {
            'version': ver, 'accepted': acc, 'held': held, 'quarantined': quar,
            'tp': tp, 'precision': round(prec, 4), 'recall': round(rec, 4), 'f1': round(f1, 4),
            'wrong_links': wrong_links, 'resolvable_missing': resolvable_missing,
        }

    r1 = eval_ver('v1', c['aar_v1'])
    r2 = eval_ver('v2', c['aar_v2'])
    assert r2['wrong_links'] == 0 and r2['resolvable_missing'] == 0
    comp = [r1, r2]
    csv_out(out / 'aar_comparison.csv', comp)
    c['aar_comparison'] = comp
    return comp

# STEP 04 개발 자료로 기준을 정한 뒤 별도 자료로 평가하기
def m09_s4(c):
    """개발 자료로 기준을 정한 뒤 별도 자료로 평가하기."""
    out = folder(c, 9)
    alert_dev = read_json(c['root'] / 'data/alert_dev.json')
    grid = []
    for rel_th in (0.0, 0.02, 0.05):
        for del_th in (30, 60, 120):
            tp = fp = fn = tn = 0
            for r in alert_dev:
                pred = (r['relative_difference'] > rel_th) or (r['delay_minutes'] > del_th)
                actual = bool(r['review_required'])
                if pred and actual: tp += 1
                elif pred and not actual: fp += 1
                elif not pred and actual: fn += 1
                else: tn += 1
            prec = tp / (tp + fp) if (tp + fp) else 0.0
            rec = tp / (tp + fn) if (tp + fn) else 0.0
            f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
            grid.append({'rel_diff_th': rel_th, 'delay_th_min': del_th, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'F1': round(f1, 4)})
    best = sorted(grid, key=lambda x: (-x['F1'], x['FP'], x['rel_diff_th'], x['delay_th_min']))[0]
    csv_out(out / 'tuning_results.csv', grid)

    # 별도 평가 자료 예측을 정답 확인 전에 먼저 고정 저장
    aar_holdout = read_json(c['root'] / 'data/aar_holdout.json')
    alert_holdout = read_json(c['root'] / 'data/alert_holdout.json')
    aar_preds = [c['mapping_fn'](r, mode='v2') for r in aar_holdout]
    alert_preds = [
        {'case_id': r['case_id'], 'predicted_review_required': (r['relative_difference'] > best['rel_diff_th']) or (r['delay_minutes'] > best['delay_th_min'])}
        for r in alert_holdout
    ]
    frozen = {'best_alert_threshold': best, 'aar_holdout_predictions': aar_preds, 'alert_holdout_predictions': alert_preds}
    write_json(out / 'frozen_predictions.json', frozen)
    csv_out(out / 'holdout_results.csv', aar_preds)

    inst = c.get('instructor_root')
    if inst and (inst / 'aar_key.json').is_file() and (inst / 'alert_key.json').is_file():
        aar_key = {r['case_id']: r for r in read_json(inst / 'aar_key.json')}
        alert_key = {r['case_id']: r['review_required'] for r in read_json(inst / 'alert_key.json')}
        aar_ok = sum(1 for r in aar_preds if r['status'] == aar_key[r['case_id']]['expected_status'])
        al_ok = sum(1 for r in alert_preds if r['predicted_review_required'] == alert_key[r['case_id']])
        alert_eval = {'status': 'SCORED', 'aar_correct': aar_ok, 'alert_correct': al_ok, 'alert_total': len(alert_preds)}
    else:
        alert_eval = {'status': 'NOT_SCORED', 'reason': '교관용 holdout 정답표 미제공(정답 분리 보관)', 'prediction_count': len(alert_preds)}
    write_json(out / 'alert_holdout_results.json', alert_eval)

    c['aar_holdout_evaluated'] = aar_preds
    c['alert_holdout_metrics'] = alert_eval
    c['best_alert'] = best
    return grid

# STEP 05 스무 항목의 사용 확인과 규칙 되돌리기 시험하기
def m09_s5(c):
    """스무 항목의 사용 확인과 규칙 되돌리기 시험하기."""
    out = folder(c, 9)
    uat = read_json(c['root'] / 'data/uat_checks.json')
    # v1 롤백 재현 시험
    v1_rollback = [c['mapping_fn'](r, mode='v1') for r in c['aar_dev']]
    rollback_exact = (v1_rollback == c['aar_v1'])
    assert rollback_exact

    (out / 'rollback.md').write_text(
        '# v2 -> v1 복원(Rollback) 절차 및 검증\n'
        '1. 실행 설정을 `config/mapping_v1.json`으로 전환하고 `mapping_fn(..., mode="v1")` 호출.\n'
        f'2. 개발 70건 재실행 결과 최초 v1 결과와 70/70건 완전 일치 확인(`rollback_exact={rollback_exact}`).\n'
        '3. 재적용 시 `config/mapping_v2.json` 및 원자료 SHA256 검증 후 재실행.\n',
        encoding='utf-8',
    )

    ev_ok = {
        'UAT01': (folder(c, 2) / 'ingest_manifest.json').is_file(),
        'UAT02': sum(c['status_counts'][k] for k in ('ACCEPTED', 'HELD', 'QUARANTINED')) == 2160,
        'UAT03': (folder(c, 5) / 'weight_sensitivity.csv').is_file(),
        'UAT04': any(r['raw_unit'] == 'BOX' and r['quantity_ea'] == 80 for r in c['normalized']),
        'UAT05': (folder(c, 2) / 'normalized.csv').is_file(),
        'UAT06': (folder(c, 2) / 'test_results.json').is_file(),
        'UAT07': (folder(c, 3) / 'step_01.csv').is_file(),
        'UAT08': (folder(c, 1) / 'key_contract.json').is_file(),
        'UAT09': all(r['status'] == 'ACCEPTED' for r in c['aar_holdout_evaluated'] if r['case_id'].startswith('H_normal')),
        'UAT10': (folder(c, 9) / 'root_cause.md').is_file(),
        'UAT11': c['aar_comparison'][1]['resolvable_missing'] == 0,
        'UAT12': c['aar_comparison'][1]['wrong_links'] == 0,
        'UAT13': sum(1 for r in c['aar_holdout_evaluated'] if r['case_id'].startswith('H_new_item') and r['status'] == 'HELD' and 'PENDING_NEW_KEY' in r['reasons']) == 4,
        'UAT14': (folder(c, 9) / 'frozen_predictions.json').is_file(),
        'UAT15': c['quality']['obs_denominator'] == 2160,
        'UAT16': (folder(c, 4) / 'batch_stream_diff.csv').is_file(),
        'UAT17': (folder(c, 4) / 'checkpoint.json').is_file(),
        'UAT18': c['approval'].master_count() == 0,
        'UAT19': (folder(c, 7) / 'tamper_test.json').is_file(),
        'UAT20': rollback_exact,
    }
    uat_rows = []
    for item in uat:
        cid = item['check_id']
        ok = bool(ev_ok.get(cid, False))
        uat_rows.append({
            'check_id': cid, 'check': item['check'], 'critical': item['critical'],
            'status': 'PASS' if ok else 'FAIL', 'points': item['points'] if ok else 0,
            'evidence': item['evidence_hint'],
        })
    csv_out(out / 'uat_results.csv', uat_rows)
    (out / 'review_notes.md').write_text(
        '# 미션9 검토 기록 (review_notes.md)\n'
        '- 개발 70건에서 v1 대비 v2는 정상 보고 보존, 허용 변화(S1 패딩·DEPOT-1 별칭·점 구분 날짜·미집계 결측) 해결(resolvable_missing=0, wrong_links=0), 신규 품목(S901 등) HELD/PENDING_NEW_KEY 유지 확인.\n'
        '- holdout 평가 30건·경보 12건 예측을 frozen_predictions.json에 선저장한 뒤 평가(교관 키 미제공 시 NOT_SCORED).\n'
        f"- UAT 20항목 결과 및 v1 복원 재실행 일치(rollback_exact={rollback_exact}) 확인.\n",
        encoding='utf-8',
    )
    return uat_rows

result_html = run_mission(c, 9, [m09_s1, m09_s2, m09_s3, m09_s4, m09_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
